In [ ]:
import os
from pydantic_settings import BaseSettings


class Settings(BaseSettings):
    OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
    EMBEDDING_MODEL: str = "text-embedding-3-small"
    LLM_MODEL: str = "gpt-4o-mini"
    CHUNK_SIZE: int = 400
    CHUNK_OVERLAP: int = 50
    VECTOR_DB_DIR: str = "./chroma_db"
    RETRIEVER_K: int = 4

    RESPONSE_TEMPLATE = """
    Answer the question based only on the following context:
    {context}

    Question: {question}

    Answer: {answer}

    Make sure to answer in a concise manner,and if you don't know the answer, just say "I Dont't Know."
    """

    class Config:
        env_file = ".env"


settings = Settings()


In [ ]:
import logging
from typing import List
from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader
from langchain_core.documents import Document

logger = logging.getLogger(__name__)


class FileLoader:
    def __init__(self, file_path: str) -> None:
        self.file_path = file_path

    def load(self) -> List[Document]:
        try:
            if self.file_path.endswith(".txt"):
                loader = loader = TextLoader(self.file_path)
            elif self.file_path.endswith(".pdf"):
                loader = PyPDFLoader(self.file_path)
            elif self.file_path.endswith(".docx"):
                loader = Docx2txtLoader(self.file_path)
            else:
                raise TypeError(f"Unsupported file type: {self.file_path}")

            documents = loader.load()
            print(f"Loaded {len(documents)} document(s)")
            return documents

        except Exception as e:
            logger.error(f"Error loading {self.file_path}: {e}")
            raise


In [ ]:
# from typing import List
# from langchain_openai import OpenAIEmbeddings
# import numpy as np


# def single_embedding(embedding: OpenAIEmbeddings, document: str):
#     emb = embedding.embed_query(document)
#     print(f"Vector Dimension: {len(emb)}")
#     print(f"First 5 Values: {emb[:5]}")
#     print(f"Vector Norm: {np.linalg.norm(emb):.4f}")
#     return emb


# def batch_embedding(embedding: OpenAIEmbeddings, document: List):
#     embeddor = embedding.embed_documents(document)

#     try:
#         for i, emb in enumerate(embeddor):
#             print(f"Text {i+1} - Vector Dimension: {len(emb)}")
#             print(f"Text {i+1} - First 5 Value: {emb[:5]}")
#             print(f"Text {i+1} - Vector Norm: {np.linalg.norm(emb):.4f}")
#             return emb
#     except:
#         raise Exception


# class Embeddor:
#     def __init__(self, document: str | List) -> None:
#         self.document = document
#         self.embedding = OpenAIEmbeddings(model="text-embedding-3-small")

#     def embeded(self):
#         try:
#             if isinstance(self.document, str):
#                 return single_embedding(self.embedding, self.document)
#             elif isinstance(self.document, List):
#                 return batch_embedding(self.embedding, self.document)
#             else:
#                 raise TypeError("ERR: Type Error!")
#         except Exception as e:
#             raise Exception(f"EMB ERR: {e}")


In [ ]:
from typing import List
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
import logging

from langchain_openai import OpenAIEmbeddings

logger = logging.getLogger(__name__)
load_dotenv()


class VectorStoreManager:
    def __init__(self) -> None:
        self.embeddings = OpenAIEmbeddings(model=settings.EMBEDDING_MODEL)
        self.persist_directory = settings.VECTOR_DB_DIR

    def create_db(self, documents: List[Document]):
        logger.info(f"Creating DB in with {len(documents)} Chunks.")
        vector_store = Chroma.from_documents(
            documents=documents,
            embedding=self.embeddings,
            persist_directory=self.persist_directory,
            collection_name="enterprise_hybrid_search",
        )
        logger.info("Vector store created successfully.")
        return vector_store

    def get_db(self):
        return Chroma(
            persist_directory=self.persist_directory,
            embedding_function=self.embeddings,
            collection_name="enterprise_hybrid_search",
        )


In [ ]:
from typing import List, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
import logging

logger = logging.getLogger(__name__)


def recursive_chunking(document: List, chunk_size: int = 400, chunk_overlap: int = 50):
    return RecursiveCharacterTextSplitter(
        separators=["\n\n", "\n", ". ", " ", ""],
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    ).split_documents(document)


def semantic_chunking(document: List[Document], embedding: Any):
    return SemanticChunker(
        embedding,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=90,
    ).split_documents(document)


class DocumentChunker:
    def __init__(self, document: List, semantic: bool = True, embeddor=None) -> None:
        self.semantic = semantic
        self.document = document
        self.embeddor = embeddor

    def chunk(self):
        try:
            if self.semantic:
                chunks = semantic_chunking(self.document, self.embeddor)
            else:
                chunks = recursive_chunking(self.document)

            logger.info(f"\nSemantic Chunks: {len(chunks)}\n")
            for i, chunk in enumerate(chunks):
                logger.info(f"\nChunk {i+1}: ({len(chunk.page_content)} chars)\n")
                logger.info(
                    chunk.page_content[:100]
                    + ("..." if len(chunk.page_content) > 100 else "")
                )
            return chunks
        except Exception as e:
            raise Exception(f"[CHUNKING ERROR]: {e}")


In [ ]:
import logging
from typing import List
from langchain_chroma import Chroma
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from pydantic import Field

logger = logging.getLogger(__name__)


class HybridRetriever(BaseRetriever):
    vector_retriever: Any = Field(description="The Chroma vector retriever")
    bm25_retriever: Any = Field(description="The BM25 keyword retriever")
    k: int = 4
    bm25_weight: float = 0.5
    vector_weight: float = 0.5
    rrf_k: int = 60

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> List[Document]:
        logger.info(f"Performing hybrid search for query: '{query}'")

        bm25_results = self.bm25_retriever.invoke(query)
        vector_results = self.vector_retriever.invoke(query)

        doc_scores = {}

        def add_to_scores(results, weight):
            for rank, doc in enumerate(results):
                key = doc.page_content
                rrf_score = weight * (1.0 / (rank + self.rrf_k))
                if key in doc_scores:
                    doc_scores[key] = (doc_scores[key][0] + rrf_score, doc)
                else:
                    doc_scores[key] = (rrf_score, doc)

        add_to_scores(bm25_results, self.bm25_weight)
        add_to_scores(vector_results, self.vector_weight)

        sorted_docs = sorted(doc_scores.values(), key=lambda x: x[0], reverse=True)
        return [doc for _, doc in sorted_docs[: self.k]]

    @classmethod
    def from_documents(
        cls,
        vector_store: Chroma,
        documents: List[Document],
        k: int = 4,
        bm25_weight: float = 0.5,
    ):
        logger.info("Initializing Hybrid Retriever...")
        vector_retriever = vector_store.as_retriever(search_kwargs={"k": k})
        bm25_retriever = BM25Retriever.from_documents(documents)
        bm25_retriever.k = k
        return cls(
            vector_retriever=vector_retriever,
            bm25_retriever=bm25_retriever,
            k=k,
            bm25_weight=bm25_weight,
            vector_weight=1.0 - bm25_weight,
        )


In [ ]:
# from typing import List
# from langchain_chroma import Chroma
# from langchain_classic.retrievers.document_compressors import LLMChainExtractor
# from langchain_classic.retrievers import ContextualCompressionRetriever
# from langchain_core.documents import Document
# import tempfile
# from dotenv import load_dotenv
# from langchain_openai import OpenAIEmbeddings

# load_dotenv()


# def contextual_compression(document: List[Document], llm, vector_db: Chroma):
#     print("=" * 60)
#     print("CONTEXTUAL COMPRESSION")
#     print("Extract only relevant parts of a long document")
#     print("=" * 60)

#     compression = LLMChainExtractor.from_llm(llm)
#     compression_retriever = ContextualCompressionRetriever(
#         base_compressor=compression,
#         base_retriever=vector_db.as_retriever(search_kwargs={"k": 2}),
#     )

#     print(f"\nContent: {document[0].page_content}\nMetadata: {document[0].metadata}")
#     compressed_docs = compression_retriever.invoke(document[0].page_content)
#     print(f"\n-- With Compression (relavant only) ---")
#     for doc in compressed_docs:
#         print(f"\nLength: {len(doc.page_content)} chars")
#         print(f"\nContent: {doc.page_content[:150]}...\n")


# def create_base_vector_store(documents):
#     return Chroma.from_documents(
#         documents=documents,
#         embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
#         persist_directory=tempfile.mkdtemp(),
#     )


# class ContextCompressor:
#     def __init__(self, document: List[Document], llm) -> None:
#         self.document = document
#         self.llm = llm

#     def compress(self):
#         return contextual_compression(
#             self.document,
#             self.llm,
#             create_base_vector_store(self.document),
#         )


In [ ]:
import logging
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

logger = logging.getLogger(__name__)


class RAGChainManager:
    def __init__(self, retriever):
        self.retriever = retriever
        self.llm = ChatOpenAI(model=settings.LLM_MODEL, temperature=0)
        self.prompt = ChatPromptTemplate.from_template(settings.RESPONSE_TEMPLATE)

    def _format_docs(self, docs):
        return "\n\n".join(doc.page_content for doc in docs)

    def build_chain(self):
        logger.info("Building LCEL RAG chain...")
        return (
            {
                "context": self.retriever | self._format_docs,
                "question": RunnablePassthrough(),
            }
            | self.prompt
            | self.llm
            | StrOutputParser()
        )


In [ ]:
import langchain
import numpy
import pandas
from dotenv import load_dotenv

from enterprise_rag.ingestion.loaders import DocumentLoaderFactory

load_dotenv()

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def setup_rag_system(file_path: str):
    logger.info(f"Initializing Enterprise RAG System for {file_path}")

    # 1. Load Document
    documents = DocumentLoaderFactory.load(file_path)

    # 2. Chunking
    chunker = DocumentChunker(documents)
    chunks = chunker.chunk()

    # 3. Vector Store Setup
    db_manager = VectorStoreManager()
    vector_store = db_manager.create_db(chunks)

    # 4. Hybrid Retriever Setup
    retriever = HybridRetriever.from_documents(
        vector_store=vector_store,
        documents=chunks,
        k=settings.RETRIEVER_K,
    )

    # 5. Build RAG Chain
    chain_manager = RAGChainManager(retriever=retriever)
    rag_chain = chain_manager.build_chain()

    return rag_chain


def checker():
    print("LangChain:", langchain.__version__)
    print("NumPy:", numpy.__version__)
    print("Pandas:", pandas.__version__)


if __name__ == "__main__":
    checker()

    try:
        rag_chain = setup_rag_system("data/doc.txt")

        print("\n--- RAG System Ready ---")
        while True:
            question = input("\nAsk a question (or type 'quit' to exit): ")
            if question.lower() in ["quit", "exit", "q"]:
                break

            response = rag_chain.invoke(question)
            print(f"\nResponse:\n{response}")

    except Exception as e:
        logger.error(f"Failed to run RAG system: {e}")
